In [1]:
from datasets import load_dataset
from tqdm import tqdm
import os, gc

OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)
TARGET_LINES = 1_000_000

def download_gujarati():
    dataset = load_dataset(
        "ai4bharat/IndicCorpV2", "indiccorp_v2",
        split="guj_Gujr", streaming=True, trust_remote_code=True,
    )
    sentences = []
    with tqdm(total=TARGET_LINES, desc="Gujarati") as pbar:
        for sample in dataset:
            text = sample.get("text", "").strip()
            if text:
                sentences.append(text)
                pbar.update(1)
            if len(sentences) >= TARGET_LINES:
                break
    with open(os.path.join(OUTPUT_DIR, "gujarati.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(sentences))
    print(f"Saved {len(sentences)} Gujarati lines.")

def download_english(target_lines=1_000_000, flush_every=50_000):
    dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)
    path = os.path.join(OUTPUT_DIR, "english.txt")
    count = 0
    buffer = []
    with open(path, "w", encoding="utf-8") as f, tqdm(total=target_lines, desc="English (C4)") as pbar:
        for sample in dataset:
            text = sample.get("text", "").strip()
            if text:
                buffer.append(text)
                count += 1
                pbar.update(1)
            if len(buffer) >= flush_every:
                f.write("\n".join(buffer) + "\n")
                buffer.clear()
                gc.collect()
            if count >= target_lines:
                break
        if buffer:
            f.write("\n".join(buffer) + "\n")
    print(f"Saved {count} English lines.")


# download_english()

In [2]:
import re

URL_RE   = re.compile(r'(https?://\S+|www\.\S+)')
EMAIL_RE = re.compile(r'[\w\.-]+@[\w\.-]+\.\w+')
DATE_RE  = re.compile(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b')
NUM_RE   = re.compile(r'\b\d+(?:[.,]\d+)*\b')

SENT_SPLIT_RE = re.compile(r'(?<=[।॥.!?])\s+')
PUNCT_RE = re.compile(r'([.,!?;:"\'()\[\]{}—–\-…]|।|॥)')

def sentence_tokenize(text):
    text = text.strip()
    if not text:
        return []
    sentences = SENT_SPLIT_RE.split(text)
    return [s.strip() for s in sentences if s.strip()]

def word_tokenize(sentence):
    tokens = []
    protected = []
    def protect(pattern, s):
        def repl(m):
            protected.append(m.group(0))
            return f' __PROT{len(protected)-1}__ '
        return pattern.sub(repl, s)

    s = protect(URL_RE, sentence)
    s = protect(EMAIL_RE, s)
    s = protect(DATE_RE, s)

    s = PUNCT_RE.sub(r' \1 ', s)

    s = re.sub(r'(\d)\s*\.\s*(\d)', r'\1.\2', s)

    for tok in s.split():
        if tok.startswith('__PROT') and tok.endswith('__'):
            idx = int(tok[6:-2])
            tokens.append(protected[idx])
        else:
            tokens.append(tok)
    return tokens

In [3]:
def process_corpus(input_path, output_prefix):
    with open(input_path, encoding="utf-8") as f:
        text = f.read()

    all_sentences = []
    for para in text.split("\n"):
        all_sentences.extend(sentence_tokenize(para))

    with open(f"{output_prefix}_sentences.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(all_sentences))

    with open(f"{output_prefix}_tokens.txt", "w", encoding="utf-8") as f:
        for sent in all_sentences:
            toks = word_tokenize(sent)
            f.write(" ".join(toks) + "\n")

    return all_sentences

In [6]:
def compute_stats(sentences):
    all_tokens = []
    total_chars = 0
    for sent in sentences:
        toks = word_tokenize(sent)
        all_tokens.extend(toks)
        total_chars += sum(len(t) for t in toks)

    num_sentences = len(sentences)
    num_words = len(all_tokens)
    types = set(all_tokens)

    stats = {
        "total_sentences": num_sentences,
        "total_words": num_words,
        "total_characters": total_chars,
        "avg_sentence_length": num_words / num_sentences if num_sentences else 0,
        "avg_word_length": total_chars / num_words if num_words else 0,
        "type_count": len(types),
        "token_count": num_words,
        "type_token_ratio": len(types) / num_words if num_words else 0,
    }
    return stats

for name in ["gujarati"]:
    sents = process_corpus(f"data/{name}.txt", f"data/{name}")
    stats = compute_stats(sents)
    print(f"\n--- {name.upper()} STATS ---")
    for k, v in stats.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")


--- GUJARATI STATS ---
total_sentences: 2833
total_words: 47691
total_characters: 211930
avg_sentence_length: 16.8341
avg_word_length: 4.4438
type_count: 13747
token_count: 47691
type_token_ratio: 0.2883


In [7]:
# NLTK
import nltk
import spacy
nltk.download('punkt')
from nltk.tokenize import sent_tokenize, word_tokenize as nltk_word_tokenize

def nltk_stats(text):
    sents = sent_tokenize(text)
    words = [w for s in sents for w in nltk_word_tokenize(s)]
    types = set(words)
    return {
        "sentences": len(sents),
        "words": len(words),
        "avg_sent_len": len(words)/len(sents),
        "ttr": len(types)/len(words),
    }

nlp_en = spacy.load("en_core_web_sm")

def spacy_stats(text, nlp):
    doc = nlp(text)
    sents = list(doc.sents)
    words = [t.text for t in doc if not t.is_space]
    types = set(words)
    return {
        "sentences": len(sents),
        "words": len(words),
        "avg_sent_len": len(words)/len(sents),
        "ttr": len(types)/len(words),
    }

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


## Lab 4

In [8]:
import random
from collections import defaultdict, Counter
import math

def load_sentences(path):
    sentences = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            toks = line.strip().split()
            if toks:
                sentences.append(toks)
    return sentences

sentences = load_sentences("data/gujarati_tokens.txt")
sentences = sentences[:1000000]
print(len(sentences))

2833


In [10]:
random.seed(42)
random.shuffle(sentences)

test_data = sentences[:1000]
dev_data = sentences[1000:2000]
train_data = sentences[2000:]

print(len(train_data), len(dev_data), len(test_data))

833 1000 1000


In [11]:
def add_boundaries(sents, n):
    padded = []
    for s in sents:
        toks = ["<s>"] * (n - 1) + s + ["</s>"]
        padded.append(toks)
    return padded

In [12]:
class NGramModel:
    def __init__(self, n):
        self.n = n
        self.ngram_counts = defaultdict(Counter)
        self.context_counts = Counter()
        self.vocab = set()

    def train(self, sents):
        padded = add_boundaries(sents, self.n)
        for toks in padded:
            self.vocab.update(toks)
            for i in range(self.n - 1, len(toks)):
                context = tuple(toks[i - self.n + 1:i])
                word = toks[i]
                self.ngram_counts[context][word] += 1
                self.context_counts[context] += 1

    def prob(self, context, word):
        V = len(self.vocab)
        c_context = self.context_counts.get(context, 0)
        c_ngram = self.ngram_counts.get(context, {}).get(word, 0)
        return (c_ngram + 1) / (c_context + V)

    def sentence_logprob(self, sent):
        toks = ["<s>"] * (self.n - 1) + sent + ["</s>"]
        logprob = 0.0
        for i in range(self.n - 1, len(toks)):
            context = tuple(toks[i - self.n + 1:i])
            word = toks[i]
            p = self.prob(context, word)
            logprob += math.log(p)
        return logprob

    def perplexity(self, sents):
        total_logprob = 0.0
        total_words = 0
        for s in sents:
            total_logprob += self.sentence_logprob(s)
            total_words += len(s) + 1
        return math.exp(-total_logprob / total_words)

In [13]:
unigram = NGramModel(1)
bigram = NGramModel(2)
trigram = NGramModel(3)
quadrigram = NGramModel(4)

In [14]:
unigram.train(train_data)

In [15]:
bigram.train(train_data)

In [16]:
trigram.train(train_data)
quadrigram.train(train_data)

In [17]:
models = {"unigram": unigram, "bigram": bigram, "trigram": trigram, "quadrigram": quadrigram}

for name, model in models.items():
    dev_pp = model.perplexity(dev_data)
    test_pp = model.perplexity(test_data)
    print(name, "dev:", dev_pp, "test:", test_pp)

unigram dev: 1807.1309143460553 test: 1875.682185894832
bigram dev: 2713.08801513103 test: 2797.307193148997
trigram dev: 4242.308765784395 test: 4290.55253414636
quadrigram dev: 5183.411386278077 test: 5215.497683361122


In [18]:
context = ("આ",)
word = "છે"
print(bigram.prob(context, word))

0.00034340659340659343


## Lab 5

In [20]:
class NGramModel:
    def __init__(self, n):
        self.n = n
        self.ngram_counts = defaultdict(Counter)
        self.context_counts = Counter()
        self.vocab = set()

    def train(self, sents):
        padded = add_boundaries(sents, self.n)
        for toks in padded:
            self.vocab.update(toks)
            for i in range(self.n - 1, len(toks)):
                context = tuple(toks[i - self.n + 1:i])
                word = toks[i]
                self.ngram_counts[context][word] += 1
                self.context_counts[context] += 1

    def prob_laplace(self, context, word):
        V = len(self.vocab)
        c_context = self.context_counts.get(context, 0)
        c_ngram = self.ngram_counts.get(context, {}).get(word, 0)
        return (c_ngram + 1) / (c_context + V)

    def prob_addk(self, context, word, k):
        V = len(self.vocab)
        c_context = self.context_counts.get(context, 0)
        c_ngram = self.ngram_counts.get(context, {}).get(word, 0)
        return (c_ngram + k) / (c_context + k * V)

    def sentence_logprob(self, sent, method="laplace", k=0.3):
        toks = ["<s>"] * (self.n - 1) + sent + ["</s>"]
        logprob = 0.0
        for i in range(self.n - 1, len(toks)):
            context = tuple(toks[i - self.n + 1:i])
            word = toks[i]
            if method == "laplace":
                p = self.prob_laplace(context, word)
            else:
                p = self.prob_addk(context, word, k)
            logprob += math.log(p)
        return logprob

    def perplexity(self, sents, method="laplace", k=0.3):
        total_logprob = 0.0
        total_words = 0
        for s in sents:
            total_logprob += self.sentence_logprob(s, method, k)
            total_words += len(s) + 1
        return math.exp(-total_logprob / total_words)

In [21]:
unigram = NGramModel(1)
bigram = NGramModel(2)
trigram = NGramModel(3)
quadrigram = NGramModel(4)

In [22]:
unigram.train(train_data)

In [23]:
bigram.train(train_data)

In [24]:
trigram.train(train_data)
quadrigram.train(train_data)

In [25]:
models = {"unigram": unigram, "bigram": bigram, "trigram": trigram, "quadrigram": quadrigram}

In [26]:
K = 0.3

for name, model in models.items():
    dev_pp = model.perplexity(dev_data, method="addk", k=K)
    test_pp = model.perplexity(test_data, method="addk", k=K)
    print(name, "dev:", dev_pp, "test:", test_pp)

unigram dev: 2248.055805456796 test: 2355.0682949037837
bigram dev: 2183.5668972908966 test: 2276.55629041992
trigram dev: 3812.8455074289054 test: 3868.9168709486535
quadrigram dev: 4950.320240881039 test: 4990.90498819525


In [27]:
for name, model in models.items():
    lap_dev = model.perplexity(dev_data, method="laplace")
    addk_dev = model.perplexity(dev_data, method="addk", k=K)
    lap_test = model.perplexity(test_data, method="laplace")
    addk_test = model.perplexity(test_data, method="addk", k=K)
    print(name)
    print("laplace dev:", lap_dev, "addk dev:", addk_dev)
    print("laplace test:", lap_test, "addk test:", addk_test)

unigram
laplace dev: 1807.1309143460553 addk dev: 2248.055805456796
laplace test: 1875.682185894832 addk test: 2355.0682949037837
bigram
laplace dev: 2713.08801513103 addk dev: 2183.5668972908966
laplace test: 2797.307193148997 addk test: 2276.55629041992
trigram
laplace dev: 4242.308765784395 addk dev: 3812.8455074289054
laplace test: 4290.55253414636 addk test: 3868.9168709486535
quadrigram
laplace dev: 5183.411386278077 addk dev: 4950.320240881039
laplace test: 5215.497683361122 addk test: 4990.90498819525
